# Blackjack with Monte Carlo

## Goal
I want to measure how good a Blackjack strategy is by **playing hands and averaging what happened**, without ever being told a single card probability. In my Dynamic Programming project I built the whole model by hand and calculated the answer. Here the agent just plays.

This notebook covers both halves of Monte Carlo. **Prediction** asks how good a fixed policy is. **Control** goes further and finds the best policy.

## Where Monte Carlo sits

| | Needs the model? | When does it learn? |
|---|---|---|
| Dynamic Programming | Yes | Never plays, just calculates |
| Monte Carlo | No | After each **full episode** |
| Temporal Difference | No | After each **single step** |

Most real problems have no model. Nobody can write down the exact probabilities of a market, which is why this matters for my XAU/USD project later.

## The core idea
> To find out how good something is, try it many times and average the results.

For Blackjack: instead of calculating that a hard 20 against a dealer 6 is worth about +0.70, I play thousands of hands that pass through that situation, write down how each one ended, and average. After enough hands the average lands near the true value.

I already think like this in trading: I do not derive the expected profit of a strategy from a formula, I backtest many trades and look at the average.

**The one catch:** Monte Carlo can only learn from episodes that **end**. A Blackjack hand always ends and a trade always closes, so both work.

## My state, actions and rewards
The state is `(player_sum, dealer_up_card, usable_ace)`.

* `player_sum`: the current total of my hand
* `dealer_up_card`: the one dealer card I can see (1 means an ace, 10 covers 10, J, Q, K)
* `usable_ace`: `1` if my hand has an ace currently counted as 11, otherwise `0`. It is a yes or no flag, not a count of aces.

Actions: **stick (0)** or **hit (1)**. Rewards: win +1, draw 0, lose -1, and nothing is won or lost until the hand ends.

## The policy I am evaluating
A fixed, simple rule: **stick if my total is 20 or more, otherwise hit.** It is a fairly weak strategy, which makes it a good test case. Prediction does not improve it, it only measures it.

## How this notebook is built
I build it the way I learn it: **small pieces first, then one big loop where everything comes together.**

1. New tools in the code: `sum_hand`, `env.unwrapped`, `card_name`.
2. Play **one** hand and record it, like at a real table.
3. Turn the record into returns: reward vs return, and gamma.
4. Store the returns per state: `returns_sum` and `returns_count`.
5. Scale up from one hand to many hands, in four small stages.
6. The full loop, with a million hands.
7. Read the estimates and watch them settle.
8. Control: a Q table for every state and action, a greedy policy, and epsilon greedy exploration.
9. Test the finished policy over many hands, then watch it play on screen.

In [1]:
import random
import time

import numpy as np
import gymnasium as gym
from gymnasium.envs.toy_text.blackjack import sum_hand

## Step 1a: Adding up a hand with `sum_hand`

To print what is happening at the table, I need the total of a hand. `sum_hand(cards)` is Gymnasium's own function for this, and it handles aces for me. Its rule: add up all the cards, then add 10 more if an ace can count as 11 without busting. It is the same logic as my `ace_value` from the Dynamic Programming project.

| Cards | `sum_hand` | Why |
|---|---|---|
| `[1, 6]` | 17 | the ace counts as 11, because 11 + 6 = 17 does not bust |
| `[1, 6, 10]` | 17 | the ace as 11 would bust at 27, so it counts as 1 |
| `[1, 1]` | 12 | one ace counts as 11, the other as 1 |
| `[10, 5, 9]` | 24 | no ace, a plain sum: a bust |

In [2]:
# the four hands from the table above, to see the ace logic with my own eyes
print("sum_hand([1, 6]) =", sum_hand([1, 6]))        # expect 17
print("sum_hand([1, 6, 10]) =", sum_hand([1, 6, 10]))    # expect 17
print("sum_hand([1, 1]) =", sum_hand([1, 1]))        # expect 12
print("sum_hand([10, 5, 9]) =", sum_hand([10, 5, 9]))    # expect 24

sum_hand([1, 6]) = 17
sum_hand([1, 6, 10]) = 17
sum_hand([1, 1]) = 12
sum_hand([10, 5, 9]) = 24


## Step 1b: `env.unwrapped`, the cards hidden inside the game

`gym.make(...)` does not give me the Blackjack game directly. It gives me the game **inside a box**: a thin wrapper that only checks I use the game properly (for example, `reset()` before `step()`). The box hides the cards, so `env.player` fails with `AttributeError: 'OrderEnforcing' object has no attribute 'player'`.

`env.unwrapped` takes the box off and gives me the real game, which holds:

| Expression | What it holds |
|---|---|
| `env.unwrapped.player` | my cards, for example `[6, 4, 2]` |
| `env.unwrapped.dealer` | the dealer's cards, for example `[4, 10, 10]` |
| `env.unwrapped.dealer[0]` | the dealer's up card, the one he shows me |

**Important:** this is peeking inside the game, which a real player cannot do. So I use it **only for printing**. The policy chooses actions from `curr_state` alone, which is exactly what the agent is allowed to know.

In [4]:
env = gym.make("Blackjack-v1")

print("type of env (the box) :", type(env))
print("\ntype of env.unwrapped (the real game):", type(env.unwrapped))

curr_state, info = env.reset()    # deal a hand so there are cards to look at

# trying to reach the cards through the box fails, so I catch the error to see its message
try:
    print(env.player)
except AttributeError as error:
    print("\nenv.player fails with :", error)

print("\nmy cards (env.unwrapped.player) :", env.unwrapped.player)
print("dealer's cards (env.unwrapped.dealer):", env.unwrapped.dealer)
print("dealer's up card (dealer[0]) :", env.unwrapped.dealer[0])
print("the state that reset() gave me :", curr_state, " (player_sum, dealer_card, usable_ace)")

type of env (the box) : <class 'gymnasium.wrappers.common.OrderEnforcing'>

type of env.unwrapped (the real game): <class 'gymnasium.envs.toy_text.blackjack.BlackjackEnv'>

env.player fails with : 'OrderEnforcing' object has no attribute 'player'

my cards (env.unwrapped.player) : [3, 10]
dealer's cards (env.unwrapped.dealer): [9, 7]
dealer's up card (dealer[0]) : 9
the state that reset() gave me : (13, 9, 0)  (player_sum, dealer_card, usable_ace)


## Step 1c: `[-1]` and `card_name`

**`[-1]`** means "the last item in a list" in Python. When I hit, the new card is added to the **end** of my hand, so `env.unwrapped.player[-1]` is the card I just drew. After a stick I draw nothing, so there is no card to show. Only the dealer's cards change, because he now plays out his hand.

**`card_name`** is for display only. The environment stores an ace as `1` and 10, J, Q and K all as `10`, so I cannot tell which face card was drawn. The function turns the numbers into readable names. It plays no part in the learning.

In [5]:
# [-1] on a plain list: the last item
example_hand = [6, 4, 2]
print("example hand :", example_hand)
print("\nexample_hand[-1] (last card) :", example_hand[-1])    # expect 2, the card that was drawn last


# helper for DISPLAY ONLY: turns a card number into a readable name
def card_name(card):
    if card == 1:
        return "Ace"
    elif card == 10:
        return "10 (or J/Q/K)"     # the environment stores 10, J, Q and K all as 10, so I cannot tell which
    else:
        return str(card)


print("\ncard_name(1) =", card_name(1))
print("card_name(10) =", card_name(10))
print("card_name(7) =", card_name(7))

example hand : [6, 4, 2]

example_hand[-1] (last card) : 2

card_name(1) = Ace
card_name(10) = 10 (or J/Q/K)
card_name(7) = 7


## Step 2: Play one hand like it happens at a real table

The environment never tells me which card was drawn, so I read the hands from inside the game for display. What I **record** for learning is still only `(state, action, reward)` per step, because that is all the agent is allowed to know:

```
[((11, 10, 0), 1, 0.0), ((13, 10, 0), 1, -1.0)]
```

Read the first entry as: "I was in state `(11, 10, 0)`, I hit (action 1), and I got reward 0.0."

Two things to get right in this loop:

* **Record the OLD state.** The `episode.append(...)` line must come **before** `curr_state = next_state`. Once `curr_state` is replaced, the state I was in when I chose the action is gone.
* **The policy reads only `curr_state[0]`**, my hand total. It is a plain `if` and `else`, no dictionary needed.

First the setup: an empty record, the flags that say when the hand is over, and the deal.

In [18]:
# set up one hand

episode = []           # (state, action, reward) per step: all the AGENT is allowed to know
terminated = False     # becomes True when the hand ends
truncated = False      # time limit flag, always False in Blackjack

curr_state, info = env.reset()    # deal a new hand: (player_sum, dealer_card, usable_ace)

print("NEW HAND")
# env.unwrapped.player shows my first 2 cards
print(f"\nour cards : {env.unwrapped.player}  (total {sum_hand(env.unwrapped.player)})")
# env.unwrapped.dealer[0] shows the dealer's first card (the up card)
print(f"\ndealer shows : {card_name(env.unwrapped.dealer[0])}")
print(f"\nstarting state : {curr_state}  (player_sum, dealer_card, usable_ace)")

NEW HAND

our cards : [9, 8]  (total 17)

dealer shows : 10 (or J/Q/K)

starting state : (17, 10, 0)  (player_sum, dealer_card, usable_ace)


Now the hand itself. Each pass of the loop is one step: choose the action from the policy, take it, record it, and print what happened at the table.

In [19]:
# play the hand step by step

step_number = 0

while not (terminated or truncated):

    # policy: stick (0) on 20 or more, otherwise hit (1)
    if curr_state[0] >= 20:
        curr_action = 0
    else:
        curr_action = 1

    next_state, reward, terminated, truncated, info = env.step(curr_action)

    # record the OLD state, the action taken in it, and the reward received
    # this must happen BEFORE curr_state is replaced at the bottom of the loop
    episode.append((curr_state, curr_action, reward))

    step_number = step_number + 1

    print(f"\nSTEP {step_number}")
    print(f"state before : {curr_state}")

    if curr_action == 1:   # hit
        # [-1] is the last card in the list: the one just drawn, i.e. my most recent card
        drawn_card = env.unwrapped.player[-1]
        print(f"action : HIT")
        print(f"card drawn : {card_name(drawn_card)}")
        print(f"our cards : {env.unwrapped.player}  (total {sum_hand(env.unwrapped.player)})")
    else:                  # stick
        # a stick draws no card for me, so instead the dealer plays out his hand and I can reveal it
        print(f"action : STICK")
        print(f"dealer reveals : {env.unwrapped.dealer}  (total {sum_hand(env.unwrapped.dealer)})")

    print(f"state after : {next_state}")
    print(f"reward : {reward}")

    curr_state = next_state

# the final result in words (reward still holds the last step's reward here)

if reward == 1:
    print("\nRESULT: you win")
elif reward == -1:
    print("\nRESULT: you lose")
else:
    print("\nRESULT: draw")


print("\nrecorded episode, as (state, action, reward) per step:")
print(episode)


STEP 1
state before : (17, 10, 0)
action : HIT
card drawn : 2
our cards : [9, 8, 2]  (total 19)
state after : (19, 10, 0)
reward : 0.0

STEP 2
state before : (19, 10, 0)
action : HIT
card drawn : 3
our cards : [9, 8, 2, 3]  (total 22)
state after : (22, 10, 0)
reward : -1.0

RESULT: you lose

recorded episode, as (state, action, reward) per step:
[((17, 10, 0), 1, 0.0), ((19, 10, 0), 1, -1.0)]


### Reading this output

* Every step before the last one has `reward = 0.0`. Nothing is won or lost until the hand ends.
* The **last** step carries the result: +1.0 for a win, 0.0 for a draw, -1.0 for a loss. A bust shows a total above 21 in `state after`.
* If the policy stuck, I see `dealer reveals`, because that is when the dealer plays. If it hit, I see `card drawn`.
* Run the cell again for a new hand. Hands can be one step long (dealt a 20 and stuck) or several.

## Step 3a: Reward vs return

This is the idea that is easiest to mix up. These two are different things:

| | Question it answers |
|---|---|
| **Reward** | What did I get **right now**, for this one step? |
| **Return** (`G`) | What did I get from this step **until the end of the episode**? |

**Worked example.** I hit on a 10, land on 19, hit again and bust:

| Step | State | Reward | Return |
|---|---|---|---|
| 1 | (10, 10, 0) | 0.0 | **-1.0** |
| 2 | (19, 10, 0) | -1.0 | **-1.0** |

* Step 2 is the last step, so its return is just its own reward: -1.0.
* Step 1 got a reward of 0.0 itself, but its return is `0.0 + (-1.0) = -1.0`, because it was part of a story that ended in a loss.

That is why `-1.0` appears **twice** in the output. Both steps belong to the same hand, and they share its ending. It is also why a state whose own reward was 0.0 can still have a return of -1.0.

The code below runs that exact example, with the rewards of the two steps side by side with their returns.

In [21]:
# the two rewards of the example hand: step 1 got 0.0, step 2 (the bust) got -1.0
rewards_example = [0.0, -1.0]    # gamma is 1.0 here, so a later reward counts fully and there is no shrinking

for t in range(len(rewards_example)):

    G = 0.0
    for k in range(t, len(rewards_example)):     # all rewards from step t to the end
        G = G + rewards_example[k]

    print(f"step {t + 1}: reward = {rewards_example[t]}, return = {G}")

step 1: reward = 0.0, return = -1.0
step 2: reward = -1.0, return = -1.0


### Is it fair that hitting on a 10 gets -1?
Hitting on a 10 was a perfectly good move. The bad luck came afterwards. One hand is a **noisy sample**, not a verdict on the state. Over thousands of hands, `(10, 10, 0)` will sometimes get +1, sometimes 0 and sometimes -1, and the **average** settles on the true value under my policy. That is exactly why Monte Carlo averages instead of trusting a single hand.

## Step 3b: What gamma really does

A tempting reading is that gamma controls "how much the immediate step matters". It is close but not quite. `gamma` (the discount factor) is **how much a reward that arrives later is worth compared to a reward now**:

* `gamma = 1.0`: a later reward counts fully, the same as one now.
* `gamma = 0.9`: every extra step of waiting shrinks the reward to 90% of its value.
* `gamma` near 0: only the immediate reward matters.

The return formula, with gamma raised to a growing power:

$$G = r_1 + \gamma\, r_2 + \gamma^2\, r_3 + \gamma^3\, r_4 + \dots$$

Example with `gamma = 0.9`, a 3 step hand whose rewards are 0, 0 and +1:

| Step | Steps until the end | Return |
|---|---|---|
| first | 3 | 0.9 × 0.9 × 1 = 0.81 |
| second | 2 | 0.9 × 1 = 0.90 |
| third | 1 | 1.00 |

Steps closer to the ending get more of the credit. The code below uses the powers of gamma exactly as in the formula: for step `t`, the first reward gets `gamma^0` (which is 1), the next `gamma^1`, then `gamma^2`.

It is somewhat very similar to the vanishing gradient problem we face in neural networks and RNN's.

In [22]:
rewards_example = [0.0, 0.0, 1.0]     # a 3 step hand that ends with a win
gamma_example = 0.9

for t in range(len(rewards_example)):

    G = 0.0
    power = 0                          # the first reward counts with gamma^0 = 1

    for k in range(t, len(rewards_example)):
        G = G + (gamma_example ** power) * rewards_example[k]
        power = power + 1              # the next reward is one step further away

    print(f"step {t + 1}: return with gamma = {gamma_example} is {G:.2f}")    # expect 0.81, 0.90, 1.00

step 1: return with gamma = 0.9 is 0.81
step 2: return with gamma = 0.9 is 0.90
step 3: return with gamma = 0.9 is 1.00


**For Blackjack I use `gamma = 1.0`.** A hand is short, and winning on the third card is not worth less than winning on the first. Every step gets full credit for the final result.

## Step 3c: The return of every step in the real hand

A return needs the whole ending, so this runs **after** the hand is over, below the `while` loop, never inside it.

For step `t`, the inner loop adds up the rewards of step `t`, `t+1`, `t+2` and so on to the end, each multiplied by its power of gamma. It is the same code as the demo above, now running on the real `episode`.

Two things worth understanding here:

* **For step `t`, only the steps from `t` to the end count.** The inner loop runs over exactly that range. Wrapping another loop around it, one that visits every step again for each step, would process the same reward over and over: a hand of 2 steps would add each reward **twice** and give `-2.0` instead of `-1.0`. One pass per step is the rule.
* **Keep the name `reward` for the real last reward.** The result printing reads `reward` after the loop. Unpacking each entry into a variable also called `reward` would silently replace it, which is why the entry is unpacked as `step_reward` below.

In [24]:
# return of every step, using gamma to the power 0, 1, 2, ...
# this runs AFTER the hand is over, because a return needs the whole ending

gamma = 1.0            # no shrinking: every step of a hand gets full credit for the result

returns = []           # returns[t] is the return of step t, same order as episode

for t in range(len(episode)):

    G = 0.0            # this step's return starts at zero
    power = 0          # the first reward counts with gamma^0 = 1

    # only the steps from t to the end matter for step t's return
    for k in range(t, len(episode)):
        state, action, step_reward = episode[k]      # step_reward, so the real reward above is not overwritten
        G = G + (gamma ** power) * step_reward
        power = power + 1                            # the next reward is one step further away

    returns.append(G)

print(f"returns of each step (gamma = {gamma}): {returns}")

returns of each step (gamma = 1.0): [-1.0, -1.0]


### The backwards shortcut

Instead of powers of gamma, I can walk the episode **backwards** and use one line:

```
G = reward + gamma * G
```

`G` already holds the return of everything after the current step, so each step back is automatically discounted a bit more. With `gamma = 1.0` it is the same as `G = G + reward`. Both methods must give identical numbers, so this is a good check on the code above.

In [25]:
returns_backward = [0.0] * len(episode)    # same length as episode, filled in from the end

G = 0.0
for t in reversed(range(len(episode))):    # walk from the last step back to the first
    state, action, step_reward = episode[t]
    G = step_reward + gamma * G            # this step's reward plus the discounted return of what came after
    returns_backward[t] = G

print("returns using powers of gamma :", returns)
print("returns walking backwards :", returns_backward)    # must match the line above

returns using powers of gamma : [-1.0, -1.0]
returns walking backwards : [-1.0, -1.0]


## Step 4a: What Monte Carlo stores

One hand is a noisy sample, so I need to average over many. To average thousands of returns without keeping every single one, I only need two numbers **per state**:

1. `returns_sum`: the running total of all returns seen for that state
2. `returns_count`: how many times I have visited that state

The estimate is `V(state) = returns_sum / returns_count`.

**Worked example.** State `(10, 10, 0)` is visited in 4 hands, with returns +1, -1, +1 and 0. Then `returns_sum = 1`, `returns_count = 4`, and the estimate is `1 / 4 = 0.25`.

In [26]:
returns_seen = [1, -1, 1, 0]      # the four returns state (10, 10, 0) received in four different hands

demo_sum = 0.0
demo_count = 0

for G in returns_seen:
    demo_sum = demo_sum + G        # add this hand's return to the running total
    demo_count = demo_count + 1    # one more visit

print("returns_sum :", demo_sum)                  # expect 1.0
print("returns_count :", demo_count)                # expect 4
print("estimated V :", demo_sum / demo_count)     # expect 0.25

returns_sum : 1.0
returns_count : 4
estimated V : 0.25


### First visit vs every visit
If the same state shows up twice in one episode, I have two returns for it. Two choices:

* **First visit:** use only the return from the first time the state appeared.
* **Every visit:** use the return from every appearance. This was my instinct: more samples should cancel the noise.

Both converge to the right answer. A small catch with every visit: two returns from the same episode share the same ending, so they are **not independent** and cancel noise less than two separate episodes would.

**For Blackjack it makes no difference.** A state can never repeat inside one hand: a hit always raises my total, or turns a soft ace into a hard one, which is a different state. So first visit and every visit give exactly the same numbers here. It will matter later, in a game where I can wander back to the same place, and in the trading project.

## Step 4b: Adding this hand's returns into the dictionaries

Both dictionaries are keyed by **state**, the same way my `V` was in the Dynamic Programming project, and they start empty. For every step of the hand, find its state and its return, then:

1. If the state is **new** (not yet in the dictionaries), give it a starting entry: a total of `0.0` and a count of `0`. This must come **before** the additions, or Python raises a `KeyError` when I try to add to a key that does not exist.
2. Add the return to `returns_sum[state]`.
3. Add 1 to `returns_count[state]`.

Two things to get right here:

* **This loop runs after the hand, outside and below the `while` loop.** Placed inside it, it would run after every step on a half finished hand, using returns that do not exist yet.
* **Two lines do the actual storing**: the addition to `returns_sum` and the `+ 1` in `returns_count`. The "new state" check only creates the empty entries, so nothing is stored until those two lines run.

In [27]:
# add this hand's returns into returns_sum and returns_count

returns_sum = {}       # state -> running total of all returns seen for that state
returns_count = {}     # state -> how many times that state has been visited

for t in range(len(episode)):

    state, action, step_reward = episode[t]          # which state this step was in
    G = returns[t]                                   # the return that step earned

    # first time this state is ever seen: give it a starting entry in both dictionaries
    # this must come BEFORE the additions below, or Python raises a KeyError
    if state not in returns_sum:
        returns_sum[state] = 0.0
        returns_count[state] = 0

    returns_sum[state] = returns_sum[state] + G      # add this return to the running total
    returns_count[state] = returns_count[state] + 1  # one more visit to this state

### What the dictionaries hold

After one hand, every state of that hand appears once, with `visits = 1` and a total return equal to that step's return. With `gamma = 1.0`, every state in a hand that ended in a bust or a loss shows `-1.0`, and every state in a win shows `1.0`.

The dictionaries are created in the cell above, so running it again resets them. In the next section they will be created once, before a loop over many hands, so they survive across hands.

In [30]:
# show what the dictionaries now hold

print("returns_sum and returns_count after this hand:")
for state in returns_sum:
    print(f"\nstate {state}: total return = {returns_sum[state]}, visits = {returns_count[state]}")
    print(f"estimated V = total / visits = {returns_sum[state] / returns_count[state]}")

returns_sum and returns_count after this hand:

state (17, 10, 0): total return = -1.0, visits = 1
estimated V = total / visits = -1.0

state (19, 10, 0): total return = -1.0, visits = 1
estimated V = total / visits = -1.0


## Step 5: From one hand to many hands

One hand is a noisy sample, so the real work is playing thousands of hands and letting the dictionaries grow across all of them. Every piece I need already exists above. What changes is **where each piece lives**: some things are created once and survive, others restart for every hand.

I build it in small stages, and each stage is a cell that runs on its own:

| Stage | What gets added | What I check |
|---|---|---|
| 5a | the things that live **outside** the hand loop | they start empty |
| 5b | the hand loop: play many hands, no learning yet | one entry per hand |
| 5c | the returns of each hand | the three return rules |
| 5d | the dictionary update | visits equal steps |
| Step 6 | everything together, a million hands | the same checks, at scale |

## Step 5a: What lives where

Before writing the loop, the question is which things must **survive** from one hand to the next and which must **start fresh** every hand.

| Where | What | Why |
|---|---|---|
| Created **once**, above the hand loop | `env`, `gamma`, `num_hands`, `returns_sum`, `returns_count`, `total_rewards`, `total_steps` | these collect knowledge across all hands |
| Reset at the **top of every hand** | `episode`, `terminated`, `truncated`, `curr_hand_rewards`, `curr_hand_steps`, and `curr_state` from `env.reset()` | these describe one hand only |
| Rebuilt **after every hand** | `returns` | it must line up with that hand's `episode` |

Two points that save trouble later:

* **If the dictionaries were created inside the loop, every new hand would wipe what the earlier hands taught them**, and they would only ever hold the last hand. They belong above the loop.
* **A variable that is reset inside the loop does not also need to be set above it.** Setting `episode = []` in both places is harmless but redundant, and it hides which things really have to survive.

I also pick one spelling for each name and use it everywhere. With `returns_sum` and `returns_count`, a slip like `return_sum` raises a `NameError`, because Python treats it as a brand new name.

In [31]:
# Created ONCE, above the hand loop: these survive across all hands 
env = gym.make("Blackjack-v1")

gamma = 1.0                 # a later reward is worth the same as now in Blackjack
num_hands = 10              # how many hands to play (small for now, scaled up later)

returns_sum = {}            # state -> running total of all returns seen for that state
returns_count = {}          # state -> number of times that state was visited

total_rewards = []          # one entry per hand: the sum of rewards in that hand
total_steps = []            # one entry per hand: the number of steps in that hand

# a labeled look at the starting point: everything must be empty
print("returns_sum starts with", len(returns_sum), "states")
print("returns_count starts with", len(returns_count), "states")
print("total_rewards starts with", len(total_rewards), "entries")
print("total_steps starts with", len(total_steps), "entries")
print("gamma =", gamma)
print("num_hands =", num_hands)

returns_sum starts with 0 states
returns_count starts with 0 states
total_rewards starts with 0 entries
total_steps starts with 0 entries
gamma = 1.0
num_hands = 10


## Step 5b: The hand loop, playing many hands

Now the loop over hands. Inside it, the top of every hand resets what belongs to that hand, and then the same `while` loop as before plays it out. There is no learning yet, only playing and counting.

Three things to get right:

* **`curr_state[0]`, not `curr_state`.** The state is a tuple like `(14, 10, 0)`, and a tuple cannot be compared with the number 20 (Python raises a `TypeError`). The player sum is the first item, so the policy reads `curr_state[0]`.
* **Record the old state, then move on.** `episode.append(...)` comes before `curr_state = next_state`, as in the single hand version.
* **The two `append` lines run once per hand, after the `while` loop ends.** The question to ask is: at which indentation does "this hand is finished" happen? Placed inside the `while` loop, they would run after every step and record a 3 step hand three times.

**What to check:** entries in `total_steps` should hold exactly `num_hands` entries (one per hand, not one per step), and the average number of steps per hand tells me how long a typical hand is under this policy.

In [32]:
# Created ONCE, above the hand loop 
env = gym.make("Blackjack-v1")
gamma = 1.0
num_hands = 10

returns_sum = {}
returns_count = {}
total_rewards = []
total_steps = []

# Hand loop: everything inside is reset for each new hand
for hand_number in range(num_hands):

    # fresh start for this hand
    episode = []                      # list of (state, action, reward) for this hand
    terminated = False
    truncated = False
    curr_hand_rewards = 0.0
    curr_hand_steps = 0
    curr_state, info = env.reset()    # deal a new hand

    # play one hand until it ends
    while not (terminated or truncated):

        if curr_state[0] >= 20:       # the player sum is the FIRST item of the state tuple
            curr_action = 0           # stick
        else:
            curr_action = 1           # hit

        next_state, reward, terminated, truncated, info = env.step(curr_action)

        episode.append((curr_state, curr_action, reward))   # record the OLD state
        curr_hand_rewards = curr_hand_rewards + reward
        curr_hand_steps = curr_hand_steps + 1
        curr_state = next_state                             # move on only after recording

    # the hand is finished: these two run ONCE per hand, after the while loop
    total_rewards.append(curr_hand_rewards)
    total_steps.append(curr_hand_steps)

# labeled checks
print("Hands recorded in total_steps (expect", num_hands, "):", len(total_steps))
print("Steps of each hand:", total_steps)
print("Reward of each hand:", total_rewards)

steps_added = 0
for s in total_steps:
    steps_added = steps_added + s
print("Average steps per hand:", steps_added / len(total_steps))

Hands recorded in total_steps (expect 10 ): 10
Steps of each hand: [3, 1, 4, 3, 1, 1, 1, 1, 2, 1]
Reward of each hand: [1.0, 0.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 0.0]
Average steps per hand: 1.8


Hands are short, usually one to four steps: either I am dealt something close to 20, or a couple of hits settle it. That is also why a million hands is cheap to run later.

## Step 5c: Adding the returns to the loop

The return calculation from Step 3c now goes **inside the hand loop, after the `while` loop**, because a return needs the whole ending of that hand. The code is the same as before. Only its position and one reset matter:

* **`returns = []` must be created inside the hand loop, right before the return calculation.** If it sits above the loop, it never resets: by hand 10 it holds the returns of all ten hands stacked together, and `returns[t]` no longer matches `episode[t]`. The dictionary update that comes next pairs those two by position, so a stale list would silently attach the wrong return to the wrong state.
* **`step_reward`, not `reward`.** The unpacking inside the return loop must not overwrite the real `reward` of the hand.

To see that it works without flooding the screen, I print the check for **hand 0 only**, using `if hand_number == 0:`. The three things that must hold:

1. `episode` and `returns` have the same length.
2. The **last** return equals the **last** reward, since nothing comes after it.
3. With `gamma = 1.0`, the **first** return equals the sum of all rewards in the hand. With rewards of 0 until the end, every step of a hand ends up with the same return: the final result.

That third point is the same idea as the `(10, 10, 0)` doubt: a step can have a reward of 0 and still carry a return of -1, because it belonged to a hand that was lost.

In [33]:
import gymnasium as gym

env = gym.make("Blackjack-v1")
gamma = 1.0
num_hands = 10

returns_sum = {}
returns_count = {}
total_rewards = []
total_steps = []

for hand_number in range(num_hands):

    # fresh start for this hand
    episode = []
    terminated = False
    truncated = False
    curr_hand_rewards = 0.0
    curr_hand_steps = 0
    curr_state, info = env.reset()

    # play one hand 
    while not (terminated or truncated):

        if curr_state[0] >= 20:
            curr_action = 0
        else:
            curr_action = 1

        next_state, reward, terminated, truncated, info = env.step(curr_action)

        episode.append((curr_state, curr_action, reward))
        curr_hand_rewards = curr_hand_rewards + reward
        curr_hand_steps = curr_hand_steps + 1
        curr_state = next_state

    total_rewards.append(curr_hand_rewards)
    total_steps.append(curr_hand_steps)

    # returns for this hand 
    returns = []                      # reset for every hand so returns[t] matches episode[t]

    for t in range(len(episode)):

        G = 0.0                       # this step's return starts at zero
        power = 0                     # the first reward counts with gamma^0 = 1

        # only the steps from t to the end matter for step t's return
        for k in range(t, len(episode)):
            state, action, step_reward = episode[k]      # step_reward, so the real reward is not overwritten
            G = G + (gamma ** power) * step_reward
            power = power + 1                            # the next reward is one step further away

        returns.append(G)

    # check, first hand only 
    if hand_number == 0:
        print("Episode of hand 0 (state, action, reward):", episode)
        print("Returns of hand 0, one per step:", returns)
        print("Same length (episode vs returns):", len(episode), len(returns))
        print("Last return (should equal last reward):", returns[-1])
        print("Sum of all rewards in hand 0 (should equal first return):", curr_hand_rewards)
        print("First return of hand 0:", returns[0])

Episode of hand 0 (state, action, reward): [((13, 6, 0), 1, -1.0)]
Returns of hand 0, one per step: [-1.0]
Same length (episode vs returns): 1 1
Last return (should equal last reward): -1.0
Sum of all rewards in hand 0 (should equal first return): -1.0
First return of hand 0: -1.0


## Step 5d: Adding the dictionary update

Step 4b joins the loop **after the returns, still inside the hand loop**. Every hand now feeds the dictionaries that were created once above the loop. For each step of the hand:

1. If the state is **new**, give it a starting entry in both dictionaries (`0.0` and `0`). This comes first, or Python raises a `KeyError`.
2. **Add** the return to `returns_sum[state]`. It has to be `old total + G`. Writing `returns_sum[state] = G` would run without any error but overwrite the total every time, so each state would only remember the latest return and the average would be meaningless.
3. **Add 1** to `returns_count[state]`.

**The check that proves the update loop sits in the right place:** every step of every hand adds exactly one visit to exactly one state, so the **total visits** across `returns_count` must equal the **total steps** across `total_steps`. If the two numbers differ, the update loop is in the wrong position or runs too often.

With only 10 hands, almost every state is visited just once (the number of different states is close to the number of visits), so each state's average is a single return. That is not an estimate yet, only a first look. The next stage fixes that with many more hands.

In [34]:
import gymnasium as gym

env = gym.make("Blackjack-v1")
gamma = 1.0
num_hands = 10

returns_sum = {}
returns_count = {}
total_rewards = []
total_steps = []

for hand_number in range(num_hands):

    # fresh start for this hand
    episode = []
    terminated = False
    truncated = False
    curr_hand_rewards = 0.0
    curr_hand_steps = 0
    curr_state, info = env.reset()

    # play one hand 
    while not (terminated or truncated):

        if curr_state[0] >= 20:
            curr_action = 0
        else:
            curr_action = 1

        next_state, reward, terminated, truncated, info = env.step(curr_action)

        episode.append((curr_state, curr_action, reward))
        curr_hand_rewards = curr_hand_rewards + reward
        curr_hand_steps = curr_hand_steps + 1
        curr_state = next_state

    total_rewards.append(curr_hand_rewards)
    total_steps.append(curr_hand_steps)

    # returns for this hand 
    returns = []

    for t in range(len(episode)):

        G = 0.0
        power = 0

        for k in range(t, len(episode)):
            state, action, step_reward = episode[k]
            G = G + (gamma ** power) * step_reward
            power = power + 1

        returns.append(G)

    # update the dictionaries (they live above the loop, so they persist) 
    for t in range(len(episode)):

        state, action, step_reward = episode[t]
        G = returns[t]                # the return that belongs to this step

        # first time this state is seen: start its total and count at zero
        if state not in returns_sum:
            returns_sum[state] = 0.0
            returns_count[state] = 0

        returns_sum[state] = returns_sum[state] + G          # add to the old total, do not overwrite it
        returns_count[state] = returns_count[state] + 1      # one more visit

# =labeled checks 
steps_added = 0
for s in total_steps:
    steps_added = steps_added + s
print("Total steps over all hands:", steps_added)

print("Number of different states in returns_sum:", len(returns_sum))

total_visits = 0
for state in returns_count:
    total_visits = total_visits + returns_count[state]
print("Total visits in returns_count:", total_visits)

# every step adds exactly one visit, so the two numbers must match
if total_visits == steps_added:
    print("Visits match steps: the update loop runs once per step")
else:
    print("MISMATCH: visits and steps differ, check where the update loop sits")

Total steps over all hands: 23
Number of different states in returns_sum: 22
Total visits in returns_count: 23
Visits match steps: the update loop runs once per step


## Step 6: Everything together, the full loop

Here are all the pieces in one cell, with thousands of hands. Nothing new is introduced. Read it as four parts inside one hand loop:

| Part | What it does | Built in |
|---|---|---|
| Setup (above the loop) | environment, `gamma`, dictionaries, totals | Step 5a |
| Part 1: play | reset the hand, follow the policy, record `(state, action, reward)` | Steps 2 and 5b |
| Part 2: returns | turn each step's future rewards into a return with powers of gamma | Steps 3 and 5c |
| Part 3: update | add each return into `returns_sum` and `returns_count` | Steps 4 and 5d |

Two things change from the small runs, because of the scale:

* **`num_hands` becomes 1,000,000.** The per hand check print is removed (it would be clutter at this size), and the loop prints nothing while it runs.
* **The visits equals steps check stays.** It is the one test that tells me the loop still behaves after a million hands.

This takes a short while to run, since it plays a million hands.

In [35]:
# SETUP: created ONCE, above the hand loop, so they survive across all hands
env = gym.make("Blackjack-v1")
gamma = 1.0                 # a later reward is worth the same as now in Blackjack
num_hands = 1000000         # how many hands to play

returns_sum = {}            # state -> total of all returns seen for that state
returns_count = {}          # state -> number of times that state was visited

total_rewards = []          # one entry per hand: sum of rewards in that hand
total_steps = []            # one entry per hand: number of steps in that hand

for hand_number in range(num_hands):

    
    # PART 1: play one hand following the policy "stick on 20 or more, otherwise hit"

    episode = []                      # (state, action, reward) for this hand
    terminated = False
    truncated = False
    curr_hand_rewards = 0.0
    curr_hand_steps = 0
    curr_state, info = env.reset()    # deal a new hand

    while not (terminated or truncated):

        if curr_state[0] >= 20:       # the player sum is the first item of the state tuple
            curr_action = 0           # stick
        else:
            curr_action = 1           # hit

        next_state, reward, terminated, truncated, info = env.step(curr_action)

        episode.append((curr_state, curr_action, reward))   # record the OLD state
        curr_hand_rewards = curr_hand_rewards + reward
        curr_hand_steps = curr_hand_steps + 1
        curr_state = next_state                             # move on only after recording

    total_rewards.append(curr_hand_rewards)    # once per hand, after the while loop
    total_steps.append(curr_hand_steps)

    # PART 2: the return of every step in this hand
    returns = []                      # reset for every hand so returns[t] matches episode[t]

    for t in range(len(episode)):

        G = 0.0                       # this step's return starts at zero
        power = 0                     # the first reward counts with gamma^0 = 1

        # only the steps from t to the end matter for step t's return
        for k in range(t, len(episode)):
            state, action, step_reward = episode[k]      # step_reward, so the real reward is not overwritten
            G = G + (gamma ** power) * step_reward
            power = power + 1                            # the next reward is one step further away

        returns.append(G)

    # PART 3: add this hand's returns into the dictionaries that persist
    for t in range(len(episode)):

        state, action, step_reward = episode[t]
        G = returns[t]                # the return that belongs to this step

        # first time this state is seen: start its total and count at zero
        if state not in returns_sum:
            returns_sum[state] = 0.0
            returns_count[state] = 0

        returns_sum[state] = returns_sum[state] + G          # add to the old total, do not overwrite it
        returns_count[state] = returns_count[state] + 1      # one more visit

# LABELED CHECKS after all hands
print("Hands recorded in total_steps (expect", num_hands, "):", len(total_steps))

steps_added = 0
for s in total_steps:
    steps_added = steps_added + s
print("Total steps over all hands:", steps_added)
print("Average steps per hand:", steps_added / len(total_steps))

print("Number of different states in returns_sum:", len(returns_sum))

total_visits = 0
for state in returns_count:
    total_visits = total_visits + returns_count[state]
print("Total visits in returns_count:", total_visits)

# every step adds exactly one visit, so the two numbers must match
if total_visits == steps_added:
    print("Visits match steps: the update loop runs once per step")
else:
    print("MISMATCH: visits and steps differ, check where the update loop sits")

Hands recorded in total_steps (expect 1000000 ): 1000000
Total steps over all hands: 1796062
Average steps per hand: 1.796062
Number of different states in returns_sum: 280
Total visits in returns_count: 1796062
Visits match steps: the update loop runs once per step


### Reading the checks

* **Different states: 280.** This is the same number of states as my Dynamic Programming grid. With 10 hands I saw almost one new state per visit. Now every state the policy can reach has been visited, and states repeat, which is what averaging needs.
* **Average steps per hand: about 1.8.** Most hands end quickly: a 20 or 21 sticks at once, and anything else usually settles within a hit or two.
* **Visits equal steps.** Each step adds exactly one visit, so the update loop runs once per step, in the right place.

## Step 7: Reading the estimates

The dictionaries are full, so the value estimate of a state is the total divided by the count:

$$V(\text{state}) = \frac{\text{returns\_sum[state]}}{\text{returns\_count[state]}}$$

Three habits for reading it:

* **Guard the lookup.** A state that was never visited has no key, and reading it raises a `KeyError`. `if state in returns_sum:` handles that, with an `else` that says "never visited".
* **Print the visit count beside the average.** The count says how far to trust the number: an average of 3 hands is mostly luck, an average of 15,000 hands is solid.
* **Keys are tuples, and `0` equals `False`.** The usable ace flag from the environment is `0` or `1`, but my Dynamic Programming dictionary used `False` and `True`. In Python `0 == False`, so `(20, 6, 0)` and `(20, 6, False)` are the same key. Either spelling finds the state.

I pick three states, each for a reason:

| State | Why it is interesting |
|---|---|
| `(20, 6, 0)` | both this policy and the best policy stick on 20, so it should land near the Dynamic Programming value of **0.704** |
| `(16, 10, 0)` | hitting is the best first move here (DP value **-0.569**), but my policy also keeps hitting on 17, 18 and 19 afterwards, so the two can disagree |
| `(21, 10, 0)` | sticking on 21 against a 10 is a very strong state, so a high value is expected |

In [36]:
picked_states = [(20, 6, 0), (16, 10, 0), (21, 10, 0)]

for state in picked_states:
    if state in returns_sum:
        # average return = total of returns divided by number of visits
        average = returns_sum[state] / returns_count[state]
        print("State", state, "| estimated value:", round(average, 3), "| visits:", returns_count[state])
    else:
        print("State", state, "| never visited")

State (20, 6, 0) | estimated value: 0.704 | visits: 15216
State (16, 10, 0) | estimated value: -0.681 | visits: 41080
State (21, 10, 0) | estimated value: 0.89 | visits: 30323


## Step 8: Watching the estimates settle

I ran the full loop from Step 6 with different values of `num_hands` and wrote down the same three states each time. These are the numbers from my own runs. A rerun will give slightly different values, because the cards are random.

| Hands | `(20, 6, 0)` estimate | visits | `(16, 10, 0)` estimate | visits | `(21, 10, 0)` estimate | visits |
|---|---|---|---|---|---|---|
| 100 | 1.000 | 3 | -1.000 | 2 | 1.000 | 2 |
| 1,000 | 0.143 | 14 | -0.632 | 38 | 0.875 | 24 |
| 10,000 | 0.717 | 173 | -0.678 | 413 | 0.888 | 294 |
| 100,000 | 0.686 | 1,559 | -0.692 | 4,132 | 0.895 | 3,126 |
| 1,000,000 | 0.707 | 14,929 | -0.683 | 41,519 | 0.889 | 30,518 |

**What the table shows**

* **Few visits give nonsense.** 3 visits that all happened to win produced exactly 1.0, and 14 visits produced 0.143. An average of a handful of random hands is mostly luck, which is why the visit count belongs next to every estimate.
* **More hands bring it to the Dynamic Programming answer.** At a million hands, `(20, 6, 0)` is 0.707 against the DP value of 0.704. DP used the exact card probabilities. Monte Carlo found the same number by only playing, with no model at all.
* **The noise shrinks slowly.** Going from 1,559 to 14,929 visits (about 10 times more) should cut the typical wobble from roughly 0.02 to about 0.006, a factor of about 3 (my single run happened to land even closer, at 0.003). Cutting the noise by a factor of 10 costs about 100 times more visits. That is the standard price of Monte Carlo.
* **`(21, 10, 0)` is stable early.** Its outcome is so lopsided that even a few hundred visits land close to 0.89.

**The gap that does not close: `(16, 10, 0)`.** It settles near **-0.68**, while the DP value is **-0.569**. More hands did not close this gap, so it is not noise. Monte Carlo measured the true value of **my policy** accurately. Hitting on 16 against a 10 is actually the right first move (the exact DP values are -0.569 for hit and -0.576 for stick, almost a tie), but my policy then keeps hitting on 17, 18 and 19, where it should stick, and those later mistakes pull the value down. DP's -0.569 is the value of playing every step well. So the estimate is correct, and the difference is how much the later decisions of the policy leave on the table.

## Step 9: From prediction to control

Prediction answers one question: how good is this fixed policy? It cannot answer the better question: what is the **best** policy? The reason is simple: it only ever sees the actions the policy takes. My policy hits below 20, so there is not a single sample of sticking on 16.

To compare actions I need `Q(state, action)` instead of `V(state)`: the average return after taking that action in that state. In Dynamic Programming I could work this out from `V`, because I knew the card probabilities and could look one step ahead. Monte Carlo has no model, so the only way to learn what hitting is worth is to hit there many times and average what happened. It is the same averaging as before, with one change: the dictionary key becomes the pair `(state, action)`.

| | Prediction | Control |
|---|---|---|
| Dictionary key | `state` | `(state, action)` |
| Question | how good is this policy? | which action is best in each state? |
| Needs | the policy's own actions | both actions tried in every state |

I go in small steps: first the Q table for my fixed policy, to see what is missing, then a random player to fill the table, then a greedy policy read out of it.

## Step 9a: The Q table for my fixed policy

My `episode` already stores the action next to every state, so the only change is in Part 3: build `key = (state, action)` and use it wherever `state` was the key. Parts 1 and 2 stay exactly as they were.

A small point in the checks afterwards: when I add up the visits, the loop variable and the variable inside the loop have to be the same (`key` and `returns_count[key]`). Using a leftover `state` from an earlier loop adds the same number again and again, and the visit check fails.

Then I print the Q value of **both** actions for 16 against a dealer 10.

In [37]:
env = gym.make("Blackjack-v1")
gamma = 1.0
num_hands = 100000

returns_sum = {}            # (state, action) -> total of all returns seen for that pair
returns_count = {}          # (state, action) -> number of visits to that pair
total_steps = 0

for hand_number in range(num_hands):

    # Part 1: play one hand with the fixed rule, stick on 20 or more, otherwise hit
    episode = []
    terminated = False
    truncated = False
    curr_state, info = env.reset()

    while not (terminated or truncated):

        if curr_state[0] >= 20:
            curr_action = 0           # stick
        else:
            curr_action = 1           # hit

        next_state, reward, terminated, truncated, info = env.step(curr_action)

        episode.append((curr_state, curr_action, reward))
        curr_state = next_state

    total_steps = total_steps + len(episode)

    # Part 2: the return of every step, same as before
    returns = []

    for t in range(len(episode)):

        G = 0.0
        power = 0

        for k in range(t, len(episode)):
            state, action, step_reward = episode[k]
            G = G + (gamma ** power) * step_reward
            power = power + 1

        returns.append(G)

    # Part 3: the only change, the key is now the (state, action) pair
    for t in range(len(episode)):

        state, action, step_reward = episode[t]
        key = (state, action)
        G = returns[t]

        if key not in returns_sum:
            returns_sum[key] = 0.0
            returns_count[key] = 0

        returns_sum[key] = returns_sum[key] + G
        returns_count[key] = returns_count[key] + 1

total_visits = 0
for key in returns_count:
    total_visits = total_visits + returns_count[key]

print(f"Different (state, action) pairs: {len(returns_sum)}")
print(f"Total steps: {total_steps}, total visits: {total_visits}")

check_state = (16, 10, 0)

for check_action in [0, 1]:
    key = (check_state, check_action)

    if check_action == 0:
        action_name = "stick"
    else:
        action_name = "hit"

    if key in returns_sum:
        q_value = returns_sum[key] / returns_count[key]
        print(f"State {check_state}, {action_name}: Q = {q_value:.3f} from {returns_count[key]} visits")
    else:
        print(f"State {check_state}, {action_name}: never visited")

Different (state, action) pairs: 280
Total steps: 179009, total visits: 179009
State (16, 10, 0), stick: never visited
State (16, 10, 0), hit: Q = -0.696 from 4154 visits


### Reading this output

* **280 pairs.** That is exactly the number of states, because the policy picks one action per state. No state has data for both actions.
* **Stick on 16 against a 10 was never visited.** The policy never sticks there, so the table has nothing to compare. Playing a million more hands would not change it.
* **Hit has a Q of about -0.7.** It matches the earlier `V` for this state, because hitting is the only action ever taken there.

With one side of the comparison missing, I cannot choose the better action. The fix is to make the player try both.

## Step 9b: A random player fills the whole table

The quickest way to get data for both actions is a player that chooses at random. The only change from Step 9a is the action line in Part 1: `curr_action = env.action_space.sample()` gives 0 or 1 with equal chance. Parts 2 and 3 do not change.

Then I print both Q values for three states.

In [38]:
env = gym.make("Blackjack-v1")
gamma = 1.0
num_hands = 100000

returns_sum = {}
returns_count = {}
total_steps = 0

for hand_number in range(num_hands):

    # Part 1: play one hand with a random player
    episode = []
    terminated = False
    truncated = False
    curr_state, info = env.reset()

    while not (terminated or truncated):

        curr_action = env.action_space.sample()      # 0 or 1, equal chance

        next_state, reward, terminated, truncated, info = env.step(curr_action)

        episode.append((curr_state, curr_action, reward))
        curr_state = next_state

    total_steps = total_steps + len(episode)

    # Part 2: returns
    returns = []

    for t in range(len(episode)):

        G = 0.0
        power = 0

        for k in range(t, len(episode)):
            state, action, step_reward = episode[k]
            G = G + (gamma ** power) * step_reward
            power = power + 1

        returns.append(G)

    # Part 3: update, keyed by the (state, action) pair
    for t in range(len(episode)):

        state, action, step_reward = episode[t]
        key = (state, action)
        G = returns[t]

        if key not in returns_sum:
            returns_sum[key] = 0.0
            returns_count[key] = 0

        returns_sum[key] = returns_sum[key] + G
        returns_count[key] = returns_count[key] + 1

total_visits = 0
for key in returns_count:
    total_visits = total_visits + returns_count[key]

print(f"Different (state, action) pairs: {len(returns_sum)}")
print(f"Total steps: {total_steps}, total visits: {total_visits}")

picked_states = [(20, 6, 0), (16, 10, 0), (21, 10, 0)]

for state in picked_states:
    for picked_action in [0, 1]:
        key = (state, picked_action)

        if picked_action == 0:
            action_name = "stick"
        else:
            action_name = "hit"

        if key in returns_sum:
            q_value = returns_sum[key] / returns_count[key]
            print(f"State {state}, {action_name}: Q = {q_value:.3f} from {returns_count[key]} visits")
        else:
            print(f"State {state}, {action_name}: never visited")

Different (state, action) pairs: 560
Total steps: 137362, total visits: 137362
State (20, 6, 0), stick: Q = 0.738 from 539 visits
State (20, 6, 0), hit: Q = -0.920 from 539 visits
State (16, 10, 0), stick: Q = -0.551 from 1515 visits
State (16, 10, 0), hit: Q = -0.746 from 1489 visits
State (21, 10, 0), stick: Q = 0.867 from 806 visits
State (21, 10, 0), hit: Q = -1.000 from 802 visits


### Reading this output

* **560 pairs.** That is 280 states times 2 actions, so every state now has data for both.
* **Stick values look right.** Sticking ends the hand at once, so its Q does not depend on what the player does later. Stick on 16 against a 10 comes out near -0.57, close to the exact DP value of -0.576.
* **Hit on a hard 21 is -1.000.** An ace counts as 1 there, so any card busts, and every one of those visits lost.
* **Hit values are too pessimistic.** `Q(state, hit)` means "hit now, then keep playing at random". A sloppy future makes hitting look worse than it is. Hit on 16 against a 10 shows about -0.78 here, while the DP value of playing well afterwards is -0.569.

That last point matters for the next two steps.

## Step 9c: Reading a greedy policy out of Q

For every state I compare `Q(state, stick)` with `Q(state, hit)` and keep the better action. This is called the **greedy** policy.

The details that matter:

* **One state at a time.** `key` is a `(state, action)` pair, so `state = key[0]` is the state. I skip a state that is already in `greedy_policy`.
* **Both actions need data.** Each membership test has its own `in` and looks in `returns_sum`: `(state, 0) in returns_sum and (state, 1) in returns_sum`.
* **The assignment belongs inside that `if`.** Outside it, the line would run with stale Q values from the previous state.
* **`np.argmax` takes one list.** `np.argmax([q_stick, q_hit])` returns 0 when stick is higher and 1 when hit is higher, which are exactly the action numbers. A second argument would be read as an axis. I wrap it in `int()` to store a plain 0 or 1.

In [39]:
greedy_policy = {}

for key in returns_sum:
    state = key[0]                     # the state of this key

    if state in greedy_policy:         # already handled through its other action
        continue

    if (state, 0) in returns_sum and (state, 1) in returns_sum:
        q_stick = returns_sum[(state, 0)] / returns_count[(state, 0)]
        q_hit = returns_sum[(state, 1)] / returns_count[(state, 1)]

        greedy_policy[state] = int(np.argmax([q_stick, q_hit]))      # 0 stick, 1 hit

stick_count = 0
hit_count = 0

for state in greedy_policy:
    if greedy_policy[state] == 0:
        stick_count = stick_count + 1
    else:
        hit_count = hit_count + 1

print(f"States with a greedy action: {len(greedy_policy)}")
print(f"The greedy policy sticks in {stick_count} states and hits in {hit_count}")
print("My Dynamic Programming policy sticks in 110 of 280 states")

States with a greedy action: 280
The greedy policy sticks in 198 states and hits in 82
My Dynamic Programming policy sticks in 110 of 280 states


### Reading this output

The greedy policy from the random player sticks in far too many states: roughly 190 to 210 in my runs, against 110 for DP. Two things cause that:

* **The pessimistic hit values** from the last section: a hit measured against a random future looks worse than it is, so stick wins too often.
* **Noise in thinly visited states.** The random player reaches some states only a few times, and a state with a handful of visits can flip between runs. The count moves by tens each time.

So a random player gives data for both actions, but the numbers are measured against a bad future. The cure is a player that follows its own best guess most of the time, and only occasionally tries something else.

## Step 10: Epsilon greedy

The rule for choosing an action in every step:

1. Draw a random number between 0 and 1.
2. If it is **below epsilon** (say 0.1), **explore**: pick a random action.
3. Otherwise **exploit**: pick the action with the higher Q for the current state.

With `epsilon = 0.1`, about 10% of the steps are random. Half of those random picks are the other action, so in a state where stick is clearly better I expect roughly **95% stick and 5% hit**.

Why this fixes the bias: once the player mostly follows its best guess, the future inside every hit Q is mostly good play, so hit values stop being pessimistic. And the 10% random steps keep both actions getting data everywhere, so an early wrong estimate can still be corrected.

Two details in the chooser:

* **It reads Q as it is right now.** A prebuilt `greedy_policy` dictionary does not exist while the hands are being played, because Q changes after every hand. So the function compares `q_stick` and `q_hit` from the dictionaries on the spot.
* **It needs a fallback.** In the early hands a state may have no data for one of the actions, and reading a missing key raises a `KeyError`. When either action has no data, the function picks randomly.

I test the chooser alone first, on the Q table from the random player, asking it for 16 against a 10 ten thousand times. There stick has the higher Q.

In [40]:
epsilon = 0.1


def epsilon_greedy(curr_state):

    # explore: with probability epsilon, pick a random action
    if random.random() < epsilon:
        return env.action_space.sample()

    # exploit: pick the action with the higher Q, but only if both actions have data
    if (curr_state, 0) in returns_sum and (curr_state, 1) in returns_sum:
        q_stick = returns_sum[(curr_state, 0)] / returns_count[(curr_state, 0)]
        q_hit = returns_sum[(curr_state, 1)] / returns_count[(curr_state, 1)]
        return int(np.argmax([q_stick, q_hit]))      # 0 if stick is higher, 1 if hit is higher

    # no data to compare yet, so try a random action
    return env.action_space.sample()


test_state = (16, 10, 0)
num_draws = 10000

for test_epsilon in [0.1, 0.5]:

    epsilon = test_epsilon
    stick_picks = 0
    hit_picks = 0

    for draw in range(num_draws):
        chosen_action = epsilon_greedy(test_state)

        if chosen_action == 0:
            stick_picks = stick_picks + 1
        else:
            hit_picks = hit_picks + 1

    print(f"epsilon {test_epsilon}: stick {stick_picks / num_draws:.1%}, hit {hit_picks / num_draws:.1%}")

epsilon = 0.1      # back to the value used in training

epsilon 0.1: stick 95.0%, hit 5.0%
epsilon 0.5: stick 74.7%, hit 25.3%


With `epsilon = 0.1` it picks stick about 95% of the time. With `epsilon = 0.5` it drops to about 75%: half the steps are random, and a quarter of all steps are hits by chance. The chooser does what the rule says.

## Step 10b: The full control loop

Now everything together, the same three parts as Step 6, with two changes:

* the action comes from `epsilon_greedy(curr_state)` instead of a fixed rule or `sample()`
* the dictionaries are keyed by `(state, action)`

Because the chooser reads the dictionaries on every step, the policy improves while the hands are being played: better Q values lead to better choices, which lead to better samples. After the loop I read the final greedy policy out of Q as in Step 9c.

This cell uses `epsilon_greedy` from the cell above and plays 200,000 hands.

In [41]:
env = gym.make("Blackjack-v1")
gamma = 1.0
num_hands = 200000
epsilon = 0.1               # 10% of the steps are random

returns_sum = {}            # (state, action) -> total of all returns seen for that pair
returns_count = {}          # (state, action) -> number of visits to that pair
total_steps = 0

for hand_number in range(num_hands):

    # Part 1: play one hand, every action chosen by epsilon greedy
    episode = []
    terminated = False
    truncated = False
    curr_state, info = env.reset()

    while not (terminated or truncated):

        curr_action = epsilon_greedy(curr_state)

        next_state, reward, terminated, truncated, info = env.step(curr_action)

        episode.append((curr_state, curr_action, reward))
        curr_state = next_state

    total_steps = total_steps + len(episode)

    # Part 2: the return of every step
    returns = []

    for t in range(len(episode)):

        G = 0.0
        power = 0

        for k in range(t, len(episode)):
            state, action, step_reward = episode[k]
            G = G + (gamma ** power) * step_reward
            power = power + 1

        returns.append(G)

    # Part 3: add the returns into the dictionaries
    for t in range(len(episode)):

        state, action, step_reward = episode[t]
        key = (state, action)
        G = returns[t]

        if key not in returns_sum:
            returns_sum[key] = 0.0
            returns_count[key] = 0

        returns_sum[key] = returns_sum[key] + G
        returns_count[key] = returns_count[key] + 1

# the final greedy policy, read out of the finished Q table
greedy_policy = {}

for key in returns_sum:
    state = key[0]

    if state in greedy_policy:
        continue

    if (state, 0) in returns_sum and (state, 1) in returns_sum:
        q_stick = returns_sum[(state, 0)] / returns_count[(state, 0)]
        q_hit = returns_sum[(state, 1)] / returns_count[(state, 1)]
        greedy_policy[state] = int(np.argmax([q_stick, q_hit]))

# checks
total_visits = 0
for key in returns_count:
    total_visits = total_visits + returns_count[key]

stick_count = 0
hit_count = 0
for state in greedy_policy:
    if greedy_policy[state] == 0:
        stick_count = stick_count + 1
    else:
        hit_count = hit_count + 1

print(f"Hands played: {num_hands}")
print(f"Total steps: {total_steps}, total visits: {total_visits}")
print(f"Different (state, action) pairs: {len(returns_sum)}")
print(f"States with a greedy action: {len(greedy_policy)}")
print(f"The greedy policy sticks in {stick_count} states and hits in {hit_count}")
print("My Dynamic Programming policy sticks in 110 of 280 states")
print()

picked_states = [(20, 6, 0), (16, 10, 0), (21, 10, 0), (12, 6, 0), (13, 2, 0)]

for state in picked_states:
    for picked_action in [0, 1]:
        key = (state, picked_action)

        if picked_action == 0:
            action_name = "stick"
        else:
            action_name = "hit"

        if key in returns_sum:
            q_value = returns_sum[key] / returns_count[key]
            print(f"State {state}, {action_name}: Q = {q_value:.3f} from {returns_count[key]} visits")
        else:
            print(f"State {state}, {action_name}: never visited")

    if greedy_policy[state] == 0:
        greedy_name = "stick"
    else:
        greedy_name = "hit"
    print(f"State {state}: the greedy policy chooses to {greedy_name}")
    print()

Hands played: 200000
Total steps: 284386, total visits: 284386
Different (state, action) pairs: 560
States with a greedy action: 280
The greedy policy sticks in 136 states and hits in 144
My Dynamic Programming policy sticks in 110 of 280 states

State (20, 6, 0), stick: Q = 0.722 from 1614 visits
State (20, 6, 0), hit: Q = -0.922 from 102 visits
State (20, 6, 0): the greedy policy chooses to stick

State (16, 10, 0), stick: Q = -0.567 from 6380 visits
State (16, 10, 0), hit: Q = -0.596 from 334 visits
State (16, 10, 0): the greedy policy chooses to stick

State (21, 10, 0), stick: Q = 0.894 from 2621 visits
State (21, 10, 0), hit: Q = -1.000 from 137 visits
State (21, 10, 0): the greedy policy chooses to stick

State (12, 6, 0), stick: Q = -0.113 from 1572 visits
State (12, 6, 0), hit: Q = -0.282 from 103 visits
State (12, 6, 0): the greedy policy chooses to stick

State (13, 2, 0), stick: Q = -0.290 from 1693 visits
State (13, 2, 0), hit: Q = -0.506 from 79 visits
State (13, 2, 0): t

### Reading this output

* **Visits equal steps.** The same check as before: every step added exactly one visit.
* **Roughly 130 to 150 stick states instead of 190 to 210.** That is much closer to the 110 of DP. The hit values are now measured against mostly good play, so the pessimism is gone.
* **The hit counts are small** (tens to a few hundred visits per state), because the player only hits about 5% of the time where sticking is better. Those are the noisiest numbers in the table, which is why the count is not exactly 110.
* **16 against a 10 is almost a tie.** DP's exact values are -0.569 for hit and -0.576 for stick, so with only a few hundred hit samples the greedy choice can come out either way, and it can flip between runs. Both are fine answers there.
* **Same loop, different job.** Nothing in Parts 2 and 3 changed from prediction. Control only changed the key and the way actions are chosen.

When I compared this policy with my DP policy state by state, roughly 90% of the 280 states agreed. The differences were mostly states where one action was barely tried, or where the two actions are nearly tied.

## Step 11: Testing the finished policy

Agreement with another table is one check. The real test is how the policy does when it plays: win rate, loss rate and the average reward per hand over many hands.

The details that matter in the test loop:

* **Pure greedy.** `greedy_policy[curr_state]` with no epsilon. Exploring was for training, and keeping it would drag the score down.
* **Counters above the loop.** `wins`, `losses`, `draws` and the reward total must be created before the hand loop. Created inside it, they reset every hand and end up holding only the last one.
* **Classify once per hand, after the `while` loop.** Most steps have a reward of 0. If the win, loss or draw check sits inside the `while` loop, every step in the middle of a hand is counted as a draw. The final `reward` is only the result once the hand is over.
* **A separate environment** (`eval_env`) and new names, so the training environment and the training variables stay untouched.

In [42]:
eval_env = gym.make("Blackjack-v1")      # separate from the training env

epochs = 100000                          # number of test hands

wins = 0
losses = 0
draws = 0
eval_total_reward = 0.0

for epoch in range(epochs):

    # only these reset for every hand
    terminated = False
    truncated = False
    curr_state, extra = eval_env.reset()

    while not (terminated or truncated):

        curr_action = greedy_policy[curr_state]      # pure greedy, no randomness

        next_state, reward, terminated, truncated, extra = eval_env.step(curr_action)
        curr_state = next_state

    # the hand is over: the last reward is the result
    eval_total_reward = eval_total_reward + reward

    if reward > 0:
        wins = wins + 1
    elif reward < 0:
        losses = losses + 1
    else:
        draws = draws + 1

print(f"Win rate:       {wins / epochs:.2%}")
print(f"Draw rate:      {draws / epochs:.2%}")
print(f"Loss rate:      {losses / epochs:.2%}")
print(f"Average reward: {eval_total_reward / epochs:.4f}")

Win rate:       43.09%
Draw rate:      8.35%
Loss rate:      48.56%
Average reward: -0.0547


### Reading this output

| Policy | Win | Draw | Loss | Average reward |
|---|---|---|---|---|
| **Monte Carlo greedy policy** | about 43% | about 9% | about 48% | **about -0.05** |
| Dynamic Programming policy | 43.3% | 9.0% | 47.7% | about -0.04 |

The Monte Carlo player never saw a card probability, and it ends up within about 0.01 to 0.02 per hand of the best possible policy. It is far better than random or always sticking.

The average reward is below zero even for the best policy, because Blackjack has a small house edge. The remaining gap comes from the handful of states where Monte Carlo picked a slightly worse action on thin data. More hands, or a smaller epsilon later in training, would close it.

## Step 12: Watching it play

Three hands of the finished policy on screen. This is a demo, not a test: three hands are mostly luck, and a strong policy still loses about half of its hands. The 100,000 hand test above is the real evidence.

* `render_mode="human"` opens a window at the first `reset()` and redraws it on every `step()`. It needs `pygame` (`pip install pygame` if gymnasium complains).
* The render environment is created **once**, outside any loop.
* `render_env.close()` at the end closes the window. On a Mac, forgetting it can leave the window hanging.
* The pauses give me time to match the table on screen with the lines printed below it.
* **The cards are printed from `env.unwrapped`**, exactly as in Step 1b: my cards, the dealer's up card, every card I draw, and the dealer's full hand when I stick. The policy only ever sees the state `(player_sum, dealer_card, usable_ace)`, so this is for reading the game, not for deciding.
* **Winning on a stick does not mean the hand was strong.** Sticking on 16 against a dealer 10 wins only when the dealer busts, which happens roughly one hand in four. The dealer's reveal line shows exactly how a win like that happens.

In [26]:
render_env = gym.make("Blackjack-v1", render_mode="human")

num_demo_hands = 3
pause_seconds = 1.5

for hand_number in range(num_demo_hands):

    terminated = False
    truncated = False
    step_number = 0
    curr_state, extra = render_env.reset()

    # peeking inside the game is for printing only, the policy never uses it
    my_cards = render_env.unwrapped.player
    dealer_cards = render_env.unwrapped.dealer

    print()
    print(f"Demo hand {hand_number + 1}")
    print(f"Your cards: {my_cards}, total {sum_hand(my_cards)}")
    print(f"Dealer shows: {card_name(dealer_cards[0])}")
    time.sleep(pause_seconds)

    while not (terminated or truncated):

        curr_action = greedy_policy[curr_state]
        step_number = step_number + 1

        next_state, reward, terminated, truncated, extra = render_env.step(curr_action)

        # read the cards again after the step, since they have just changed
        my_cards = render_env.unwrapped.player
        dealer_cards = render_env.unwrapped.dealer

        if curr_action == 1:
            print(f"Step {step_number}: state {curr_state}, the policy chooses to hit")
            print(f"  Card drawn: {card_name(my_cards[-1])}, your cards: {my_cards}, total {sum_hand(my_cards)}")
            if sum_hand(my_cards) > 21:
                print("  Bust")
        else:
            print(f"Step {step_number}: state {curr_state}, the policy chooses to stick")
            print(f"  Dealer reveals: {dealer_cards}, total {sum_hand(dealer_cards)}")
            if sum_hand(dealer_cards) > 21:
                print("  Dealer busts")

        curr_state = next_state
        time.sleep(pause_seconds)

    if reward > 0:
        result = "you win"
    elif reward < 0:
        result = "you lose"
    else:
        result = "draw"
    print(f"Result of hand {hand_number + 1}: {result} (reward {reward})")
    time.sleep(pause_seconds)

render_env.close()
print()
print("Demo finished")


Demo hand 1
Your cards: [10, 6], total 16
Dealer shows: 7
Step 1: state (16, 7, 0), the policy chooses to hit
  Card drawn: 7, your cards: [10, 6, 7], total 23
  Bust
Result of hand 1: you lose (reward -1.0)

Demo hand 2
Your cards: [10, 2], total 12
Dealer shows: 8
Step 1: state (12, 8, 0), the policy chooses to hit
  Card drawn: 5, your cards: [10, 2, 5], total 17
Step 2: state (17, 8, 0), the policy chooses to stick
  Dealer reveals: [8, 10], total 18
Result of hand 2: you lose (reward -1.0)

Demo hand 3
Your cards: [2, 10], total 12
Dealer shows: 4
Step 1: state (12, 4, 0), the policy chooses to hit
  Card drawn: 10 (or J/Q/K), your cards: [2, 10, 10], total 22
  Bust
Result of hand 3: you lose (reward -1.0)

Demo finished


## What I take away

* **Monte Carlo learns by playing and averaging.** No card probabilities, no model. It works because every hand ends, so every step gets a real return.
* **The return is the whole ending, not the reward.** A step with a reward of 0 still carries the result of the hand, and gamma decides how much a later reward counts.
* **Prediction and control share one loop.** Play, compute returns, add to the dictionaries. Control changes the key to `(state, action)` and chooses actions with epsilon greedy.
* **Both actions need data.** A fixed policy never tries the alternative, and a random player measures hitting against a bad future. Epsilon greedy fixes both.
* **Test by playing.** A win rate and an average reward over 100,000 hands says more than any single table.

## What comes next

The big project: the same averaging idea on my own XAU/USD backtested trades. Instead of playing a game many times, I resample my trade results many times (a bootstrap) and look at the spread of outcomes, so I can see how much of my backtest is skill and how much is luck.